In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os, sys
root_dir = os.path.abspath('..')

if root_dir not in sys.path:
    sys.path.append(root_dir)

from plot_utils import fast_plot

In [25]:
pd.set_option('display.max_rows', None)

In [3]:
data_path = root_dir + '/data'

In [4]:
file_path = data_path + '/task_2.xlsx'
data = pd.read_excel(file_path)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103790 entries, 0 to 103789
Data columns (total 13 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   five_min_ts                  103790 non-null  object 
 1   yandex_score                 103790 non-null  int64  
 2   median_speed_km_h            103790 non-null  int64  
 3   dtp_cnt                      103790 non-null  int64  
 4   roadworks_cnt                103790 non-null  int64  
 5   auto_cnt                     103790 non-null  int64  
 6   carsharing_vehicle_cnt       103790 non-null  int64  
 7   taxi_vehicle_cnt             103790 non-null  int64  
 8   ngpt_vehicle_cnt             103790 non-null  int64  
 9   traffic_jam_duration_common  103790 non-null  int64  
 10  travel_time                  103668 non-null  float64
 11  prec_status                  102859 non-null  object 
 12  ru_weather                   102859 non-null  object 
dtyp

In [6]:
data.head()

,five_min_ts,yandex_score,median_speed_km_h,dtp_cnt,roadworks_cnt,auto_cnt,carsharing_vehicle_cnt,taxi_vehicle_cnt,ngpt_vehicle_cnt,traffic_jam_duration_common,travel_time,prec_status,ru_weather
0,2025-01-01 00:00:00.000,0,58,0,698,3310546,14735,37510,3196,21,11.0,3мм,облачно с прояснениями
1,2025-01-01 00:05:00.000,1,58,0,649,30606,11969,26850,2532,25,11.0,3мм,облачно с прояснениями
2,2025-01-01 00:10:00.000,1,56,0,684,19609,12573,28204,2656,23,11.0,3мм,облачно с прояснениями
3,2025-01-01 00:15:00.000,1,55,0,673,26543,12371,27752,2610,24,11.0,3мм,облачно с прояснениями
4,2025-01-01 00:20:00.000,1,57,0,662,31706,12170,27301,2564,25,11.0,3мм,облачно с прояснениями


## Первичный анализ данных

In [7]:
# преобразуем ts в datetime
data['five_min_ts'] = pd.to_datetime(data['five_min_ts'])

In [9]:
columns_str = ['prec_status','ru_weather']

data['prec_status'].value_counts()

prec_status
Нет осадков    58839
0.2мм           2758
0.6мм           1503
0.4мм           1501
1.5мм           1255
               ...  
20.8 мм            1
25.3 мм            1
21.7 мм            1
25.4 мм            1
25.5 мм            1
Name: count, Length: 544, dtype: int64

In [10]:
data['ru_weather'].value_counts()

ru_weather
облачно с прояснениями    41564
пасмурно                  29619
ясно                      20936
небольшой дождь            3664
дождь со снегом            3483
дождь                      2032
небольшой снег             1185
дождь с грозой              131
сильный дождь               125
снег                         69
гроза                        51
Name: count, dtype: int64

In [ ]:
# prec_status - переведем в числовые значения
# ru_weather - категориальная фича, категории: clear, cloudy, overcast, rain_light, rain_heavy, snow, thunderstorm, other

In [16]:
def parse_prec_status(series: pd.Series) -> pd.Series:
    """
    Приводит столбец prec_status к числовым значениям (мм).
    'Нет осадков' → 0.0, '0.2мм' / '0.4 мм' / ... → float.
    """
    return (
        series
        .astype(str)
        .str.strip()
        .replace({"Нет осадков": "0", "нет осадков": "0"})
        .str.replace(r"\s*мм$", "", regex=True)   # убираем «мм» / « мм»
        .astype(float)
    )

_WEATHER_MAP = {
    # clear
    "ясно": "clear",
    # cloudy
    "облачно с прояснениями": "cloudy",
    "облачно": "cloudy",
    # overcast
    "пасмурно": "overcast",
    # rain_light
    "небольшой дождь": "rain_light",
    "морось": "rain_light",
    # rain_heavy
    "дождь": "rain_heavy",
    "сильный дождь": "rain_heavy",
    "дождь с грозой": "rain_heavy",
    # snow
    "небольшой снег": "snow",
    "снег": "snow",
    "дождь со снегом": "snow",
    # thunderstorm
    "гроза": "thunderstorm",
}


def map_weather(series: pd.Series) -> pd.Series:
    """
    Маппит ru_weather в устойчивые классы.
    Неизвестные категории → 'unknown'.
    """
    return series.str.strip().str.lower().map(_WEATHER_MAP).fillna("unknown")

In [17]:
df = data.copy()

In [18]:
df["prec_mm"] = parse_prec_status(df["prec_status"])
df["weather_class"] = map_weather(df["ru_weather"])

In [19]:
data.travel_time.describe().T

count    103668.000000
mean         14.522669
std           3.854519
min           7.000000
25%          11.000000
50%          14.000000
75%          17.000000
max         200.000000
Name: travel_time, dtype: float64

In [20]:
# travel_time можно заполнить пропуски через median, т.к. это время в пути, как правило оно не выбивается из межквартиального размаха.
median_travel_time = data.travel_time.median()
print(median_travel_time)

df['travel_time'].fillna(median_travel_time, inplace=True)

14.0


C:\Users\Amnesia\AppData\Local\Temp\ipykernel_7584\3812034788.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['travel_time'].fillna(median_travel_time, inplace=True)


In [21]:
df.travel_time.describe().T

count    103790.000000
mean         14.522054
std           3.852295
min           7.000000
25%          11.000000
50%          14.000000
75%          17.000000
max         200.000000
Name: travel_time, dtype: float64

In [22]:
df[df['prec_status'].isna()]

,five_min_ts,yandex_score,median_speed_km_h,dtp_cnt,roadworks_cnt,auto_cnt,carsharing_vehicle_cnt,taxi_vehicle_cnt,ngpt_vehicle_cnt,traffic_jam_duration_common,travel_time,prec_status,ru_weather,prec_mm,weather_class
3656,2025-01-13 16:40:00,2,40,2,686,2669054,18139,45925,4830,83,15.0,NaN,NaN,NaN,unknown
5666,2025-01-20 16:15:00,2,40,10,681,2585223,17039,43496,4749,130,16.0,NaN,NaN,NaN,unknown
5686,2025-01-20 17:55:00,4,33,11,710,2874713,16958,48427,5301,162,19.0,NaN,NaN,NaN,unknown
5687,2025-01-20 18:00:00,4,33,10,757,3080877,18070,51602,5639,167,19.0,NaN,NaN,NaN,unknown
5688,2025-01-20 18:05:00,4,32,11,728,2967399,17121,52080,5759,173,19.0,NaN,NaN,NaN,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102625,2025-12-23 12:40:00,5,33,0,757,2483342,15511,51188,5049,166,19.0,NaN,NaN,NaN,unknown
102626,2025-12-23 12:45:00,5,33,0,745,2443923,15264,50375,4969,178,19.0,NaN,NaN,NaN,unknown
102961,2025-12-24 16:55:00,8,28,0,713,3206995,17550,53467,5230,301,22.0,NaN,NaN,NaN,unknown
102984,2025-12-24 18:50:00,9,25,1,754,3561258,21646,65165,5715,299,25.0,NaN,NaN,NaN,unknown


In [28]:
df.columns

Index(['five_min_ts', 'yandex_score', 'median_speed_km_h', 'dtp_cnt',
       'roadworks_cnt', 'auto_cnt', 'carsharing_vehicle_cnt',
       'taxi_vehicle_cnt', 'ngpt_vehicle_cnt', 'traffic_jam_duration_common',
       'travel_time', 'prec_status', 'ru_weather', 'prec_mm', 'weather_class'],
      dtype='object')

In [29]:
mask = (df.five_min_ts >= '2025-01-13 16:00:00') & (df.five_min_ts < '2025-01-14')
print_cols = ['five_min_ts', 'yandex_score', 'prec_status', 'ru_weather', 'prec_mm', 'weather_class']
df.loc[mask, print_cols]

,five_min_ts,yandex_score,prec_status,ru_weather,prec_mm,weather_class
3648,2025-01-13 16:00:00,2,12.9мм,дождь со снегом,12.9,snow
3649,2025-01-13 16:05:00,2,13мм,дождь со снегом,13.0,snow
3650,2025-01-13 16:10:00,2,13мм,дождь со снегом,13.0,snow
3651,2025-01-13 16:15:00,2,12.9мм,облачно с прояснениями,12.9,cloudy
3652,2025-01-13 16:20:00,2,13мм,облачно с прояснениями,13.0,cloudy
3653,2025-01-13 16:25:00,2,13.1мм,облачно с прояснениями,13.1,cloudy
3654,2025-01-13 16:30:00,2,12.9мм,облачно с прояснениями,12.9,cloudy
3655,2025-01-13 16:35:00,2,12.9мм,облачно с прояснениями,12.9,cloudy
3656,2025-01-13 16:40:00,2,NaN,NaN,NaN,unknown
3657,2025-01-13 16:45:00,2,12.7мм,облачно с прояснениями,12.7,cloudy


# EDA